# 12 · How precisely can Lazuli measure a 1-minute He I 10830 Å emission flash on a Sun-like star: with the IFS, or with a new 5 nm WCC filter?

The He I 10830 Å triplet is a chromospheric activity diagnostic (and, in transit, a tracer of escaping exoplanet atmospheres; arXiv:1711.05269). Here a G2V star shows a **transient emission line at 10830 Å, intrinsic FWHM 0.1 Å, that lasts exactly one 1-minute exposure**. We ask for the 1σ accuracy of the line flux in that minute versus the star's z′ (AB) magnitude, with the star's own photon noise included, for:

- the **IFS** (`slicersim`), measuring the line in the R ≈ 100 spectrum;
- the **WCC** with a **new, hypothetical 5 nm top-hat filter** centred on 10830 Å, on the **qCMOS (HWK4123)** sensor (`wcc_etc` `qcmos:` noise model). The filter does not exist in `wcc_etc`; we build it.

The line is unresolved by both instruments, so what matters is its flux (or equivalent width, EW) against the noise of the star under it. The notebook ends by adapting WCC tutorial 05 (light curves): a 1-minute-cadence time series containing the flash.

**Instrument / code:** Lazuli IFS, `slicersim` (`LazuliTarget`, `change_detector`); Lazuli WCC, `wcc_etc` (`Simulation.from_sensorfilter("qcmos:z")`, `update_sensor(path_total_throughput=...)`, `get_scene("emission", host=...)`).

## Prompt to try in Jupyter AI

> A G2V star (z′ AB from 4 to 18) shows a He I 10830 Å emission line, FWHM 0.1 Å, for one 1-minute exposure.
> (1) With `slicersim.LazuliTarget`, measure the line flux in the IFS spectrum over ±1 LSF FWHM, with the star's photon noise, choosing for each magnitude a ~1-minute MACC read-out that does not saturate. Bin-integrate the line onto the IFS wavelength bins first.
> (2) With `wcc_etc`, build a 5 nm top-hat filter at 10830 Å on the `qcmos` sensor from the `qcmos:z` throughput, install it with `update_sensor(path_total_throughput=...)`, check that it took effect, and get the line S/N with `get_scene("emission", lines=[...], host="G2V")`, splitting the minute into reads to avoid saturation.
> Plot σ(EW) per minute versus z′ for both, and a 1-minute-cadence light curve with the flash at z′ = 10. State what is assumed about the continuum.

## Setup

Two gotchas, both checked below:

- **slicersim samples a custom spectrum at the centre of each IFS wavelength bin** (gallery notebook 06). A 0.1 Å line would fall between samples and silently disappear, so we integrate the line over each IFS bin analytically before handing the spectrum over.
- **The new WCC filter** is written as a throughput file in the same format as the shipped `qcmos:z` curve and installed with `update_sensor(path_total_throughput=...)`. Section 3 verifies that the ETC actually uses it before any number is computed.

In [ ]:
### use pre-cached data, just in case SVO and STScI archive servers are down for sncosmo downloads during experimentation
### for JupyterAI experimentation, feel free to uncomment this -- any newly downloaded data will go to ~/.cache/astropy/sncosmo/ and other places
import os, shutil
MOUNT = os.path.expanduser('~/s3/schmidt-observatory-system')
SRC = f'{MOUNT}/nov26-symposium/sncosmo_data_dir'
DEST = '/home/jovyan/sncosmo_data_dir'

# Copy only if it isn't already there (skip re-copying on every run)
if not os.path.exists(DEST):
    shutil.copytree(SRC, DEST)

# set sncosmo's environment variable for its default data directory
os.environ['SNCOSMO_DATA_DIR'] = DEST

import importlib
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import astropy.units as u
import matplotlib.pyplot as plt
from scipy.special import erf
import slicersim
import wcc_etc
from wcc_etc import Simulation, get_scene

print('slicersim', slicersim.__version__, '| wcc_etc', wcc_etc.__version__)
warnings.filterwarnings('ignore', message='spectrum')    # line-only spectra do not cover the whole filter: expected (WCC exercise 4)
trapezoid = getattr(np, 'trapezoid', None) or np.trapz

THUMB = Path('thumbs/12_he10830_transient_ifs_vs_wcc.png')
LINE_WAVE = 10830.0        # Å, He I triplet (air)
LINE_FWHM = 0.1            # Å, intrinsic; unresolved by both instruments
LINE_PEAK = 1.0            # line peak / local continuum: sets the *reference* line only (σ(EW) does not depend on it)
EW_REF = LINE_PEAK * LINE_FWHM * np.sqrt(np.pi / (4 * np.log(2)))     # Å; Gaussian EW = 1.0645 x FWHM x peak
T_EXP = 60.0               # s: the flash lasts one exposure
STAR = 'G2V'
MAG_BAND, MAG_SYS = 'sdss_z', 'abmag'     # z' (AB)
FILTER_WIDTH = 50.0        # Å, top hat
BASE_LABEL = 'qcmos:z'     # HWK4123 qCMOS in z: telescope x QE at 1.08 µm and the sensor noise model
Z_GRID = np.arange(4.0, 18.01, 0.25)
Z0 = 14.0                  # reference magnitude for the calibration runs
print(f"reference line: FWHM {LINE_FWHM} Å, peak {LINE_PEAK} x continuum -> EW = {EW_REF * 1e3:.0f} mÅ")

## 1. The star and the line

Both instruments see the same spectrum: the `wcc_etc` G2V (Pickles) template normalised to z′ (AB), evaluated from 3000 to 20 000 Å and handed to slicersim as an absolute flux. The line flux scales with the star: $F_\mathrm{line} = \mathrm{EW}\times F_\lambda^\mathrm{cont}(10830)$, with the continuum averaged over the 50 Å filter window.

In [ ]:
LAM = np.arange(3000.0, 20000.0, 1.0)

def star_scene(z, **kw):
    return get_scene(STAR, mag=float(z), bandpass=MAG_BAND, magsys=MAG_SYS, **kw)

_, flam = star_scene(Z0).source.get_spectrum(as_array=True, wave=LAM * u.AA, flux_unit='flam')
FLAM_Z0 = np.asarray(getattr(flam, 'value', flam), dtype=float)       # erg/s/cm2/Å at z' = Z0
assert np.all(np.isfinite(FLAM_Z0)) and np.all(FLAM_Z0 > 0), 'the G2V template does not cover 3000-20000 Å'
IN_BAND = np.abs(LAM - LINE_WAVE) <= FILTER_WIDTH / 2
FCONT_Z0 = FLAM_Z0[IN_BAND].mean()

def scale(z):
    return 10 ** (-0.4 * (np.asarray(z) - Z0))

def fcont(z):
    return FCONT_Z0 * scale(z)                # erg/s/cm2/Å under the line

def line_flux(z, ew=EW_REF):
    return ew * fcont(z)                      # erg/s/cm2

print(f"G2V at z' = {Z0}: continuum at {LINE_WAVE:.0f} Å = {FCONT_Z0:.3e} erg/s/cm2/Å; reference line flux = {line_flux(Z0):.3e} erg/s/cm2")

# cross-check the normalisation with sncosmo's SDSS z' (photon-weighted AB magnitude)
try:
    import sncosmo
    band = sncosmo.get_bandpass('sdssz')
    w = LAM[(LAM > band.minwave()) & (LAM < band.maxwave())]
    T = band(w)
    f_ab = 3631e-23 * 2.998e18 / w ** 2
    z_syn = -2.5 * np.log10(trapezoid(np.interp(w, LAM, FLAM_Z0) * T * w, w) / trapezoid(f_ab * T * w, w))
    print(f"sncosmo sdssz synthetic magnitude of the spectrum: {z_syn:.3f} (asked for {Z0})")
except Exception as err:                       # informational only
    print('sncosmo cross-check skipped:', err)

## 2. The IFS (slicersim)

The star is a point source in the narrow field. The H4RG is read up the ramp every $t_\mathrm{frame}$ = 2.83 s; one minute is ~21 frames. Bright stars need shorter ramps, repeated, which pays the read noise more often. The candidate ~1-minute read-outs are listed below; for each magnitude we take the one with the smallest error that does not saturate.

**Measurement** (as in gallery notebook 06): line signal = (spectrum with line) − (spectrum without) summed over ±1 LSF FWHM, times the bin width; noise = $\sqrt{\sum \sigma^2\,d\lambda^2}$ from the variance of the spectrum *with* the line, so the star's (and the line's) photon noise is in. Subtracting the noise-free continuum means **the continuum is assumed known** from the quiescent exposures before the flash (a long baseline; with a single reference exposure the error grows by √2).

The variance is linear in the star flux and in the line flux, so three noise-free runs per read-out (star, star + line, no star) give the error at every magnitude; a full run at two magnitudes checks this.

In [ ]:
probe = slicersim.LazuliTarget(LAM, FLAM_Z0)
LB = probe.simulation.spectrograph.lbda
EDGES = np.r_[LB[0] - (LB[1] - LB[0]) / 2, (LB[1:] + LB[:-1]) / 2, LB[-1] + (LB[-1] - LB[-2]) / 2]
IDX = np.searchsorted(EDGES, LAM)              # label i <-> IFS bin between EDGES[i-1] and EDGES[i]
NB = len(EDGES) + 1
SIG_LINE = LINE_FWHM / (2 * np.sqrt(2 * np.log(2)))

def bin_average(flux):
    mean = np.bincount(IDX, flux, minlength=NB) / np.maximum(np.bincount(IDX, minlength=NB), 1)
    return mean[IDX]

def line_per_bin(F):
    # Gaussian line of total flux F integrated exactly over each IFS bin -> bin-mean flux density (erg/s/cm2/Å)
    cdf = 0.5 * (1 + erf((EDGES - LINE_WAVE) / (np.sqrt(2) * SIG_LINE)))
    return np.r_[0.0, F * np.diff(cdf) / np.diff(EDGES), 0.0]

def line_binned(F):
    return line_per_bin(F)[IDX]                 # on the LAM grid, constant within each IFS bin

STAR_BINNED = bin_average(FLAM_Z0)
assert np.isclose(np.sum(line_per_bin(1.0)[1:-1] * np.diff(EDGES)), 1.0), 'line flux not conserved'

lr, R = probe.simulation.get_resolving_power()
R_LINE = np.interp(LINE_WAVE, lr, R)
FWHM_LSF = LINE_WAVE / R_LINE
DL = np.gradient(LB)
WIN = np.abs(LB - LINE_WAVE) <= FWHM_LSF
TFRAME = probe.get_properties('tframe')
print(f"IFS at {LINE_WAVE:.0f} Å: R = {R_LINE:.0f}, LSF FWHM = {FWHM_LSF:.0f} Å, pixel = {np.interp(LINE_WAVE, LB, DL):.0f} Å, "
      f"window ±1 FWHM = {WIN.sum()} bins; t_frame = {TFRAME} s")
print(f"the 0.1 Å line is diluted ~{FWHM_LSF / LINE_FWHM:.0f}x by the LSF")

READOUTS = [((21, 1, 0), 1), ((7, 1, 0), 3), ((3, 1, 0), 7), ((2, 1, 0), 10)]    # ~1 min each, shortest ramps last
for (n, m, d), nr in READOUTS:
    print(f"nmd = {(n, m, d)!s:11s} x {nr:2d} ramps: {nr * (n * (m + d) - d) * TFRAME:5.1f} s")

In [ ]:
def ifs_run(flux, nmd, nramps):
    # noise-free spectrum, its variance (flambda), saturation flag and exposure time
    t = slicersim.LazuliTarget(LAM, flux)
    t.change_detector(nmd=nmd, nramps=nramps)
    _, f, v = t.get_spectrum(unit='flambda', incl_error=False)
    return f, v, bool(t.simulation.detector.nsaturated_detpx), t.get_exposure_time()

def wsum(v):
    return np.sum(v[WIN] * DL[WIN] ** 2)

F0 = line_flux(Z0)
IFS_MODEL = {}
for ro in READOUTS:
    f_c, v_c, sat, texp = ifs_run(STAR_BINNED, *ro)                     # star at Z0
    f_l, v_l, _, _ = ifs_run(STAR_BINNED + line_binned(F0), *ro)         # star + line
    _, v_0, _, _ = ifs_run(STAR_BINNED * 1e-8, *ro)                       # no star: zodi + dark + read noise
    assert not sat, f'{ro} saturates at the calibration magnitude z = {Z0}'
    n0 = wsum(v_0)
    IFS_MODEL[ro] = dict(n0=n0, a=wsum(v_c) - n0, k=(wsum(v_l) - wsum(v_c)) / F0,
                         eff=np.sum((f_l - f_c)[WIN] * DL[WIN]) / F0, texp=texp)
    m = IFS_MODEL[ro]
    print(f"{ro[0]!s:11s} x {ro[1]:2d}: {texp:5.1f} s, line recovered in the window {m['eff']:.0%}, "
          f"variance at z' = {Z0}: star {m['a'] / (m['a'] + m['n0']):.0%}, detector + zodi {m['n0'] / (m['a'] + m['n0']):.0%}")

def ifs_sigma_F(z, ro, F=None):
    # 1-sigma error on the true line flux (erg/s/cm2) in one read-out, star at z'
    m = IFS_MODEL[ro]
    F = line_flux(z) if F is None else F
    return np.sqrt(m['n0'] + m['a'] * scale(z) + m['k'] * F) / m['eff']

# check the linear model against full runs
for z in (12.0, 16.0):
    F = line_flux(z)
    f_c, _, _, _ = ifs_run(STAR_BINNED * scale(z), *READOUTS[0])
    f_l, v_l, _, _ = ifs_run(STAR_BINNED * scale(z) + line_binned(F), *READOUTS[0])
    full = np.sqrt(wsum(v_l)) / np.sum((f_l - f_c)[WIN] * DL[WIN]) * F
    print(f"z' = {z}: sigma_F full run {full:.3e}, linear model {ifs_sigma_F(z, READOUTS[0]):.3e} erg/s/cm2")
    assert np.isclose(full, ifs_sigma_F(z, READOUTS[0]), rtol=0.03)

**Bright limit.** For each read-out, bisect in z′ for the magnitude where the first detector pixel saturates (gallery notebook 07's criterion, `detector.nsaturated_detpx`).

In [ ]:
def ifs_saturates(z, ro):
    return ifs_run(STAR_BINNED * scale(z), *ro)[2]

def ifs_bright_limit(ro, lo=2.0, hi=Z0, tol=0.05):
    if not ifs_saturates(lo, ro):
        return lo
    while hi - lo > tol:                         # lo saturates, hi does not
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if ifs_saturates(mid, ro) else (lo, mid)
    return hi

IFS_SAT = {ro: ifs_bright_limit(ro) for ro in READOUTS}
for ro, zs in IFS_SAT.items():
    print(f"{ro[0]!s:11s} x {ro[1]:2d}: unsaturated for z' > {zs:.2f}")

ifs_sigF = np.full(Z_GRID.size, np.nan)
ifs_ro = [None] * Z_GRID.size
for i, z in enumerate(Z_GRID):
    ok = [ro for ro in READOUTS if z > IFS_SAT[ro]]
    if ok:
        best = min(ok, key=lambda ro: ifs_sigma_F(z, ro))
        ifs_sigF[i], ifs_ro[i] = ifs_sigma_F(z, best), best
ifs_sigEW = ifs_sigF / fcont(Z_GRID)           # Å

## 3. The WCC: a new 5 nm top-hat filter on the qCMOS (HWK4123)

`qcmos:z` has total throughput = telescope × z′ filter × HWK4123 QE. The new filter is a **top hat from 10805 to 10855 Å whose transmission equals the z′ filter's at 1.08 µm**, i.e. the new total throughput is the `qcmos:z` curve inside the top hat and zero outside. The sensor (read noise, dark current, gain, well depth, minimum frame time) stays the qCMOS one.

First, what the shipped `qcmos:z` curve says at 10830 Å. If it is zero there, the throughput tables do not reach He 10830 and the filter cannot be built from them.

In [ ]:
base = Simulation.from_sensorfilter(BASE_LABEL, star_scene(Z0))
W = np.arange(9000.0, 11600.0, 0.1)
T_base = base.sensor.bandpass(W).value
T_LINE = float(base.sensor.bandpass(LINE_WAVE).value)
print(f"{BASE_LABEL}: pivot {float(base.sensor.bandpass.pivot().value):.0f} Å, total throughput at {LINE_WAVE:.0f} Å = {T_LINE:.4f} "
      f"(curve peak {T_base.max():.3f}); PSF {type(base.default_psf).__name__}")
print({k: base.sensor.meta[k] for k in ('read_noise', 'dark_current', 'well_depth', 'min_frame_time') if k in base.sensor.meta})
if not T_LINE > 1e-4:
    raise RuntimeError(f"{BASE_LABEL} throughput is ~0 at {LINE_WAVE} Å: the shipped curve does not reach He 10830. "
                       "Supply the HWK4123 QE and telescope throughput at 1.083 µm to build the filter.")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(W, T_base, color='0.4', label=BASE_LABEL)
ax.axvspan(LINE_WAVE - FILTER_WIDTH / 2, LINE_WAVE + FILTER_WIDTH / 2, color='#d1495b', alpha=0.3, label='new 5 nm top hat')
ax.set(xlabel='wavelength [Å]', ylabel='total throughput', title='qCMOS (HWK4123) z′ throughput and the new filter')
ax.legend()
plt.show()

**Build and install the filter.** Find the shipped throughput file, learn its columns, wavelength unit and scale by matching it to the installed bandpass, write the top hat in the same format, and install it with `update_sensor(path_total_throughput=...)`.

In [ ]:
def find_data_file(rel):
    rel = Path(rel)
    if rel.is_file():
        return rel
    for mod in ('lazuli_astrosim_data', 'wcc_etc'):
        try:
            root = Path(importlib.import_module(mod).__file__).parent
        except ImportError:
            continue
        hits = sorted(root.rglob(rel.name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'cannot find {rel}')

orig_path = find_data_file(base.sensor.meta['path_total_throughput'])
num = pd.read_csv(orig_path, comment='#').select_dtypes('number')
wcol, tcol = num.columns[:2]
probe_w = W[T_base > 0.2 * T_base.max()]
best = None
for unit_A in (1.0, 10.0, 1e4):                    # file in Å, nm or µm
    f = np.interp(probe_w / unit_A, num[wcol], num[tcol], left=np.nan, right=np.nan)
    ratio = f / base.sensor.bandpass(probe_w).value
    if np.all(np.isfinite(ratio)):
        spread = np.std(ratio) / np.mean(ratio)
        if best is None or spread < best[1]:
            best = (unit_A, spread, np.median(ratio))
assert best is not None and best[1] < 0.02, f'cannot match {orig_path.name} to the installed bandpass'
unit_A, _, tscale = best
print(f"{orig_path.name}: columns ({wcol}, {tcol}), wavelength unit {unit_A:g} Å, throughput scale x{tscale:.3g}")

lo, hi = LINE_WAVE - FILTER_WIDTH / 2, LINE_WAVE + FILTER_WIDTH / 2
w_file = np.asarray(num[wcol], dtype=float) * unit_A
w_new = np.unique(np.r_[w_file[(w_file < lo - 1) | (w_file > hi + 1)], lo - 1e-3, np.arange(lo, hi + 1e-9, 0.5), hi + 1e-3])
inside = (w_new >= lo) & (w_new <= hi)
t_new = np.zeros_like(w_new)
t_new[inside] = base.sensor.bandpass(w_new[inside]).value
TOPHAT_PATH = Path('wcc_qcmos_he10830_tophat_throughput.csv').resolve()
pd.DataFrame({wcol: w_new / unit_A, tcol: t_new * tscale}).to_csv(TOPHAT_PATH, index=False)

def tophat_sim(scene):
    sim = Simulation.from_sensorfilter(BASE_LABEL, scene)
    try:
        sim.update_sensor(path_total_throughput=str(TOPHAT_PATH))
    except Exception as err:
        raise RuntimeError('wcc_etc did not accept a custom throughput via update_sensor(path_total_throughput=...)') from err
    return sim

nb = tophat_sim(star_scene(Z0))
T_nb = nb.sensor.bandpass(W).value
T_want = np.where(np.abs(W - LINE_WAVE) <= FILTER_WIDTH / 2, T_base, 0.0)
edge = np.abs(np.abs(W - LINE_WAVE) - FILTER_WIDTH / 2) < 0.6
assert np.allclose(T_nb[~edge], T_want[~edge], atol=1e-3 * T_LINE), 'the installed bandpass is not the top hat'
RECT = trapezoid(T_nb, W) / float(nb.sensor.bandpass(LINE_WAVE).value)
print(f"new filter installed: pivot {float(nb.sensor.bandpass.pivot().value):.1f} Å, rectangular width {RECT:.1f} Å, "
      f"T({LINE_WAVE:.0f}) = {float(nb.sensor.bandpass(LINE_WAVE).value):.4f}")

**Does the ETC use it, and does it see a 0.1 Å line?** A line outside the top hat (10700 Å, still inside z′) must vanish, and a 0.1 Å line must give the same S/N as a 2 Å one (the filter is flat across both).

In [ ]:
def line_scene(F, wave=LINE_WAVE, fwhm=LINE_FWHM, z=None):
    kw = {} if z is None else dict(host=STAR, host_prop={'mag': float(z), 'bandpass': MAG_BAND, 'magsys': MAG_SYS})
    return get_scene('emission', mag=None, lines=[{'wave': wave, 'flux': float(F), 'fwhm': fwhm}], **kw)

def snr_line(scene):
    return tophat_sim(scene).get_image_snr(T_EXP, warn=False)['snr']

s_in, s_out, s_wide = snr_line(line_scene(1e-15)), snr_line(line_scene(1e-15, wave=10700.0)), snr_line(line_scene(1e-15, fwhm=2.0))
print(f"pure 1e-15 erg/s/cm2 line, 60 s: S/N {s_in:.1f} at 10830 Å (FWHM 0.1 Å), {s_wide:.1f} (FWHM 2 Å), {s_out:.2f} at 10700 Å")
assert s_out < 0.01 * s_in, 'a line outside the top hat is still detected: the filter is not applied'
assert np.isclose(s_in, s_wide, rtol=0.02), 'the 0.1 Å line is not sampled correctly by wcc_etc'

**Precision versus magnitude.** The line is the source and the G2V star is a point-source host under it (WCC exercise 4), so `get_image_snr` returns the line S/N with the star's photon noise. The minute is split into the fewest reads that keep every frame unsaturated (`choose_reads` from tutorial 05). The star alone in the same filter gives the classic narrowband photometric precision, which must reproduce σ(EW) = σ_rel × rectangular width.

In [ ]:
MIN_FRAME = float(base.sensor.meta.get('min_frame_time', 0.25))
READS = tuple(n for n in (1, 2, 3, 4, 5, 6, 10, 12, 15, 20, 30, 60, 120, 240) if T_EXP / n >= MIN_FRAME)

def choose_reads(sim, t, reads=READS):
    # smallest read count whose per-frame image has no saturated pixel; (None, last result) if none works (tutorial 05)
    for n in reads:
        r = sim.get_image_snr(t, n_reads=n, warn=False)
        if not np.any(r['saturated']):
            return n, r
    return None, r

wl = tophat_sim(line_scene(line_flux(Z0), z=Z0))    # line + star
ws = tophat_sim(star_scene(Z0))                     # star alone
wcc_sigF = np.full(Z_GRID.size, np.nan)
wcc_ppm = np.full(Z_GRID.size, np.nan)
wcc_reads = np.zeros(Z_GRID.size, int)
for i, z in enumerate(Z_GRID):
    F = line_flux(z)
    wl.update(host__mag=float(z))
    wl.update(source__lines=[{'wave': LINE_WAVE, 'flux': float(F), 'fwhm': LINE_FWHM}])
    n, r = choose_reads(wl, T_EXP)
    if n is None:
        continue                                     # saturates even in the fastest frames
    wcc_sigF[i], wcc_reads[i] = F / r['snr'], n
    ws.update(source__mag=float(z))
    wcc_ppm[i] = 1e6 / ws.get_image_snr(T_EXP, n_reads=n, warn=False)['snr']
assert float(wl.sensor.bandpass(LINE_WAVE + 40).value) < 1e-6, 'update() reverted the filter'
wcc_sigEW = wcc_sigF / fcont(Z_GRID)

ratio = wcc_sigEW / (wcc_ppm * 1e-6 * RECT)
print(f"brightest unsaturated z' = {Z_GRID[np.isfinite(wcc_sigF)][0]:.2f} ({wcc_reads[np.isfinite(wcc_sigF)][0]} reads per minute)")
print(f"sigma_EW (line run) / (star-only precision x {RECT:.1f} Å): median {np.nanmedian(ratio):.3f}, range {np.nanmin(ratio):.3f}-{np.nanmax(ratio):.3f}")
assert 0.9 < np.nanmedian(ratio) < 1.1

## 4. Comparison

σ(EW) per minute is independent of the line's strength wherever the star dominates the noise; the fractional accuracy of the reference line is σ(EW)/EW_ref.

In [ ]:
print(f"1-minute flash, FWHM {LINE_FWHM} Å, reference EW {EW_REF * 1e3:.0f} mÅ (peak = {LINE_PEAK} x continuum)")
print(f"{'z_AB':>5} | {'IFS sigma_EW':>12} {'S/N_ref':>7} {'read-out':>16} | {'WCC sigma_EW':>12} {'S/N_ref':>7} {'reads':>5} {'ppm':>7} | {'IFS/WCC':>7}")
for zz in (4, 6, 8, 10, 12, 14, 16, 18):
    i = np.argmin(np.abs(Z_GRID - zz))
    a, b = ifs_sigEW[i] * 1e3, wcc_sigEW[i] * 1e3
    ro = f"{ifs_ro[i][0]}x{ifs_ro[i][1]}" if ifs_ro[i] else 'saturated'
    fa = f"{a:9.2f} mÅ {EW_REF * 1e3 / a:7.1f}" if np.isfinite(a) else f"{'sat':>12} {'':>7}"
    fb = f"{b:9.2f} mÅ {EW_REF * 1e3 / b:7.1f} {wcc_reads[i]:5d} {wcc_ppm[i]:7.0f}" if np.isfinite(b) else f"{'sat':>12} {'':>7} {'':>5} {'':>7}"
    print(f"{zz:5.1f} | {fa} {ro:>16} | {fb} | {a / b:7.2f}")

## 5. A light curve with the flash (adapted from WCC tutorial 05)

Tutorial 05 draws white noise from the ETC S/N around a transit model. Here the model is the line EW: zero, except for the one exposure at t = 0. For the narrowband filter the same series in relative flux is $1 + \mathrm{EW}/W_\mathrm{rect}$. As in tutorial 05, only white noise is modelled.

In [ ]:
Z_DEMO = 10.0
i = np.argmin(np.abs(Z_GRID - Z_DEMO))
t_min = np.arange(-30, 31)                       # minutes, 1-minute cadence
ew_true = np.where(t_min == 0, EW_REF, 0.0)
rng = np.random.default_rng(3)
LC = {}
for name, sig in (('IFS', ifs_sigEW[i]), ('WCC 5 nm (qCMOS)', wcc_sigEW[i])):
    if np.isfinite(sig):
        LC[name] = (ew_true + rng.normal(0, sig, t_min.size), sig)
        quiet = LC[name][0][t_min != 0]
        print(f"{name:18s} z' = {Z_DEMO}: sigma_EW {sig * 1e3:6.2f} mÅ per minute, out-of-flash scatter {quiet.std() * 1e3:6.2f} mÅ, "
              f"flash at {EW_REF / sig:5.1f} sigma")
    else:
        print(f"{name:18s} z' = {Z_DEMO}: saturated")
if 'WCC 5 nm (qCMOS)' in LC:
    print(f"narrowband relative flux in the flash minute: 1 + {EW_REF / RECT * 1e6:.0f} ppm")

## Hero figure

In [ ]:
cols = {'IFS': '#00798c', 'WCC 5 nm (qCMOS)': '#d1495b'}
fig, (ax, bx) = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw=dict(width_ratios=[1.2, 1]))
ax.semilogy(Z_GRID, ifs_sigEW * 1e3, 'o-', ms=3, color=cols['IFS'], label='IFS (slicersim), R ≈ 100')
ax.semilogy(Z_GRID, wcc_sigEW * 1e3, 'o-', ms=3, color=cols['WCC 5 nm (qCMOS)'], label='WCC, new 5 nm top hat, qCMOS')
ax.axhline(EW_REF * 1e3 / 5, color='0.4', ls='--', lw=1)
ax.text(Z_GRID[0] + 0.2, EW_REF * 1e3 / 5 * 1.15, f'5σ for EW = {EW_REF * 1e3:.0f} mÅ (peak = continuum)', color='0.3', fontsize=11)
for arr, c in ((ifs_sigEW, cols['IFS']), (wcc_sigEW, cols['WCC 5 nm (qCMOS)'])):
    ok = np.isfinite(arr)
    if ok.any() and not ok[0]:
        ax.axvspan(Z_GRID[0], Z_GRID[ok][0], color=c, alpha=0.08)
ax.set(xlabel='G2V star, z′ (AB)', ylabel='σ(EW) in one minute [mÅ]', title='He I 10830 flash, FWHM 0.1 Å, star photon noise included')
ax.legend(loc='upper left', fontsize=11)

for k, (name, (ew, sig)) in enumerate(LC.items()):
    off = -0.6 * k * EW_REF * 1e3 if len(LC) > 1 else 0.0
    bx.errorbar(t_min, ew * 1e3 + off, yerr=sig * 1e3, fmt='o', ms=4, color=cols[name], capsize=0, lw=1, label=name)
    bx.plot(t_min, ew_true * 1e3 + off, color=cols[name], lw=1, alpha=0.6)
bx.set(xlabel='time from the flash [min]', ylabel='measured EW [mÅ] (offset)', title=f"1-minute cadence, z′ = {Z_DEMO:g}")
bx.legend(fontsize=11)
fig.tight_layout()
fig.savefig(THUMB, dpi=150)
plt.show()

## Answer

*To be filled in from the comparison table after the first run on the hub.*

## Caveats & next steps

- **The filter is hypothetical.** It is a perfect 50 Å top hat whose transmission equals the z′ filter's at 1.08 µm, on the shipped `qcmos:z` telescope × QE curve. Check the HWK4123 QE at 1083 nm against the datasheet: silicon QE falls steeply here, and the answer scales with it. Real interference filters have lower peak transmission, sloped edges, and a passband that shifts with the angle of incidence across the field.
- **Continuum knowledge.** Both instruments assume the quiescent continuum is known from the exposures before the flash (the light-curve baseline). A single narrowband filter cannot tell line emission from a continuum change in the same minute (a white-light flare would look like He I emission); the IFS measures the continuum beside the line in the same exposure. An off-line WCC filter, or a simultaneous broadband WCC image, would be the fix.
- **PSF.** `from_sensorfilter` uses the PSF of the `qcmos:z` slot; at 1.08 µm the Airy core is somewhat wider, which matters only where background and read noise do (faint end).
- **IFS measurement.** Point source in the narrow field; a box sum over ±1 LSF FWHM that loses the LSF wings (the recovered fraction is printed and corrected for), against the noise-free continuum. The fixed read-out list is a choice; `setup_to_snr` or other MACC modes may do slightly better.
- **White noise only**, as in tutorial 05: no flat-field, jitter or gain-drift floor, no saturation non-linearity or persistence.
- **Line shape.** The star's photospheric Si I 10827 Å line sits inside the filter; it is constant and cancels in the difference. Flare He I emission is often broader and shifted by tens of km/s (≲ 1 Å); both instruments still capture it.
- *Extend it:* (1) add an off-line continuum filter at 10700 Å and fold its noise in; (2) sweep the filter width from 1 to 10 nm; (3) move the star to an M dwarf, where He 10830 activity is stronger and the continuum redder; (4) put the flash in the IFS wide field.